<a href="https://colab.research.google.com/github/Oguipereira/Sistema-Bancario/blob/main/SistemaBancario_Projeto.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
%cd Sistema-Bancario

[Errno 2] No such file or directory: 'Sistema-Bancario'
/content


In [5]:
!pwd

/content


In [15]:
from dataclasses import dataclass
from datetime import datetime
from decimal import Decimal, InvalidOperation
from itertools import count
import sqlite3

# conectando o SQLITE
conexao = sqlite3.connect("banco.db")
cursor = conexao.cursor()

# Algumas exeções

class ErroBancario(Exception):
    "Exceção base do sistema bancário."
    pass


class ClienteNaoEncontradoErro(ErroBancario):
    "Cliente não encontrado na base."
    pass


class ContaNaoEncontradaErro(ErroBancario):
    "Conta não encontrada."
    pass


class ValorInvalidoErro(ErroBancario):
    "Valor inválido."
    pass


class SaldoInsuficienteErro(ErroBancario):
    "Saldo insuficiente."
    pass


class ContaBloqueadaErro(ErroBancario):
    "A conta encontra-se bloqueada."
    pass


class TransferenciaInvalidaErro(ErroBancario):
    "Transferência inválida."
    pass


# Funções auxiliares

def converter_para_decimal(valor) -> Decimal:

    #Converte um valor para Decimal com duas casas decimais

    #Exemplos:
    #1500 -> Decimal(1500.00)
    #1500,50 -> Decimal(1500.50)


    try:
        valor_formatado = str(valor).strip().replace(",", ".")
        valor_decimal = Decimal(valor_formatado)

        return valor_decimal.quantize(Decimal("0.01"))

    except (InvalidOperation, ValueError, TypeError):
        raise ValorInvalidoErro(
            "Digite um valor monetário válido."
        )


def formatar_moeda(valor: Decimal) -> str:

    #Formata o valor no padrão da moeda brasileira

    #Exemplo:
   # Decimal 1500.00 = R$ 1.500,00


    valor_formatado = f"{valor:,.2f}"

    valor_formatado = (
        valor_formatado
        .replace(",", "TEMP")
        .replace(".", ",")
        .replace("TEMP", ".")
    )

    return f"R$ {valor_formatado}"


def formatar_data(data: datetime) -> str:
    #Formata a data e a hora no padrão brasileiro

    return data.strftime("%d/%m/%Y às %H:%M:%S")


def ler_valor_monetario(mensagem: str) -> Decimal:
    #Solicita um valor ao usuário e o converte para Decimal

    valor = input(mensagem)

    return converter_para_decimal(valor)


# Modelagem de sistema

@dataclass
class Cliente:
    id: int
    nome: str
    cpf: str

    def __str__(self):
        return (
            f"ID: {self.id} | "
            f"Nome: {self.nome} | "
            f"CPF: {self.cpf}"
        )


@dataclass
class Movimentacao:
    tipo: str
    valor: Decimal
    data: datetime
    descricao: str
    saldo_apos_movimentacao: Decimal

    def __str__(self):
        return (
            f"{formatar_data(self.data)} | "
            f"{self.tipo:<25} | "
            f"{formatar_moeda(self.valor):>15} | "
            f"Saldo: "
            f"{formatar_moeda(self.saldo_apos_movimentacao):>15} | "
            f"{self.descricao}"
        )


# Classes da conta

@dataclass
class Conta:
    numero: int
    cliente: Cliente
    saldo: Decimal = Decimal("0.00")
    ativa: bool = True

    def __post_init__(self):

       # O método é executado automaticamente após a criação da conta
        #Cada conta começa com um histórico vazio


        self.historico = []

    def validar_conta_ativa(self):
        # Impede movimentações em contas bloqueadas

        if not self.ativa:
            raise ContaBloqueadaErro(
                f"A conta {self.numero} está bloqueada."
            )

    def depositar(self, valor):
        #Realiza um depósito na conta

        self.validar_conta_ativa()

        valor = converter_para_decimal(valor)

        if valor <= Decimal("0.00"):
            raise ValorInvalidoErro(
                "O valor do depósito deve ser maior que zero."
            )

        self.saldo += valor

        movimentacao = Movimentacao(
            tipo="DEPÓSITO",
            valor=valor,
            data=datetime.now(),
            descricao="Depósito realizado",
            saldo_apos_movimentacao=self.saldo
        )

        self.historico.append(movimentacao)

        print(
            f"Depósito de {formatar_moeda(valor)} "
            f"realizado com sucesso."
        )

    def sacar(self, valor):
        #Realiza um saque na conta

        self.validar_conta_ativa()

        valor = converter_para_decimal(valor)

        if valor <= Decimal("0.00"):
            raise ValorInvalidoErro(
                "O valor do saque deve ser maior que zero."
            )

        if valor > self.saldo:
            raise SaldoInsuficienteErro(
                f"Saldo insuficiente. "
                f"Saldo disponível: {formatar_moeda(self.saldo)}."
            )

        self.saldo -= valor

        movimentacao = Movimentacao(
            tipo="SAQUE",
            valor=-valor,
            data=datetime.now(),
            descricao="Saque realizado",
            saldo_apos_movimentacao=self.saldo
        )

        self.historico.append(movimentacao)

        print(
            f"Saque de {formatar_moeda(valor)} "
            f"realizado com sucesso."
        )

    def transferir(self, conta_destino, valor):
        #Transfere dinheiro desta conta para outra conta

        self.validar_conta_ativa()
        conta_destino.validar_conta_ativa()

        valor = converter_para_decimal(valor)

        if self.numero == conta_destino.numero:
            raise TransferenciaInvalidaErro(
                "Não é possível transferir para a própria conta."
            )

        if valor <= Decimal("0.00"):
            raise ValorInvalidoErro(
                "O valor da transferência deve ser maior que zero."
            )

        if valor > self.saldo:
            raise SaldoInsuficienteErro(
                f"Saldo insuficiente. "
                f"Saldo disponível: {formatar_moeda(self.saldo)}."
            )

        # Retira o dinheiro da conta de origem.
        self.saldo -= valor

        # Adiciona o dinheiro à conta de destino
        conta_destino.saldo += valor

        movimentacao_origem = Movimentacao(
            tipo="TRANSFERÊNCIA ENVIADA",
            valor=-valor,
            data=datetime.now(),
            descricao=(
                f"Transferência enviada para a conta "
                f"{conta_destino.numero}, "
                f"titular {conta_destino.cliente.nome}"
            ),
            saldo_apos_movimentacao=self.saldo
        )

        movimentacao_destino = Movimentacao(
            tipo="TRANSFERÊNCIA RECEBIDA",
            valor=valor,
            data=datetime.now(),
            descricao=(
                f"Transferência recebida da conta "
                f"{self.numero}, "
                f"titular {self.cliente.nome}"
            ),
            saldo_apos_movimentacao=conta_destino.saldo
        )

        self.historico.append(movimentacao_origem)
        conta_destino.historico.append(movimentacao_destino)

        print(
            f"Transferência de {formatar_moeda(valor)} "
            f"realizada com sucesso."
        )

    def bloquear(self):
        #Bloqueia a conta

        if not self.ativa:
            raise ContaBloqueadaErro(
                f"A conta {self.numero} já está bloqueada."
            )

        self.ativa = False

        print(f"Conta {self.numero} bloqueada com sucesso.")

    def desbloquear(self):
        #Desbloqueia a conta

        if self.ativa:
            raise ErroBancario(
                f"A conta {self.numero} já está desbloqueada."
            )

        self.ativa = True

        print(f"Conta {self.numero} desbloqueada com sucesso.")

    def consultar_saldo(self):
        #Exibe o saldo atual da conta

        print("CONSULTA DE SALDO")
        print(f"Titular: {self.cliente.nome}")
        print(f"Conta: {self.numero}")
        print(f"Saldo atual: {formatar_moeda(self.saldo)}")

    def exibir_extrato(self):
        # Exibe todas as movimentações da conta

        print(f"EXTRATO DA CONTA {self.numero}")

        print(f"Titular: {self.cliente.nome}")
        print(f"CPF: {self.cliente.cpf}")
        print(f"Status: {'ATIVA' if self.ativa else 'BLOQUEADA'}")


        if not self.historico:
            print("Nenhuma movimentação encontrada.")

        else:
            for movimentacao in self.historico:
                print(movimentacao)

        print(f"Saldo atual: {formatar_moeda(self.saldo)}")

    def __str__(self):
        status = "ATIVA" if self.ativa else "BLOQUEADA"

        return (
            f"Conta: {self.numero} | "
            f"Titular: {self.cliente.nome} | "
            f"Saldo: {formatar_moeda(self.saldo)} | "
            f"Status: {status}"
        )

# a partir daqui vamos implementar a classe do banco (contas, cliente e operações)
class Banco:

    def __init__(self):

        self.clientes = {}
        self.contas = {}

        self.proximo_cliente = count(1)
        self.proxima_conta = count(1001)

    def cadastrar_cliente(self, nome, cpf):

        id_cliente = next(self.proximo_cliente)

        cliente = Cliente(
            id=id_cliente,
            nome=nome,
            cpf=cpf
        )

        self.clientes[id_cliente] = cliente

        # Verifica se CPF já existe
        cursor.execute(
            """
            SELECT cpf
            FROM clientes
            WHERE cpf = ?
            """,
            (cpf,)
        )

        if cursor.fetchone():
            raise ValorInvalidoErro(
                "Já existe um cliente com este CPF."
            )

        cursor.execute(
            """
            INSERT INTO clientes
            (nome, cpf)

            VALUES (?, ?)
            """,
            (
                cliente.nome,
                cliente.cpf
            )
        )

        conexao.commit()

        return cliente

    def buscar_cliente(self, id_cliente):

        if id_cliente not in self.clientes:
            raise ClienteNaoEncontradoErro(
                "Cliente não encontrado."
            )

        return self.clientes[id_cliente]

    def criar_conta(self, id_cliente):

        cliente = self.buscar_cliente(id_cliente)

        numero_conta = next(self.proxima_conta)

        conta = Conta(
            numero=numero_conta,
            cliente=cliente
        )

        self.contas[numero_conta] = conta

        cursor.execute(
            """
            INSERT INTO contas
            (numero, cliente_id, saldo)

            VALUES (?, ?, ?)
            """,
            (
                numero_conta,
                cliente.id,
                float(conta.saldo)
            )
        )

        conexao.commit()

        return conta

    def buscar_conta(self, numero_conta):

        if numero_conta not in self.contas:
            raise ContaNaoEncontradaErro(
                "Conta não encontrada."
            )

        return self.contas[numero_conta]

In [16]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS clientes (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    nome TEXT NOT NULL,
    cpf TEXT NOT NULL UNIQUE
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS contas (
    numero INTEGER PRIMARY KEY,
    cliente_id INTEGER,
    saldo REAL DEFAULT 0,

    FOREIGN KEY(cliente_id)
    REFERENCES clientes(id)
)
""")

conexao.commit()

In [17]:
# teste de cadastro e criação de conta no sistema
banco = Banco()

cliente1 = banco.cadastrar_cliente(
    "Guilherme",
    "12345678900"
)

cliente2 = banco.cadastrar_cliente(
    "Joao",
    "98765432100"
)

conta1 = banco.criar_conta(cliente1.id)
conta2 = banco.criar_conta(cliente2.id)

print(conta1)
print(conta2)


Conta: 1001 | Titular: Guilherme | Saldo: R$ 0,00 | Status: ATIVA
Conta: 1002 | Titular: Joao | Saldo: R$ 0,00 | Status: ATIVA


In [18]:
# Criando dois clientes

cliente_1 = Cliente(
    id=1,
    nome="Guilherme",
    cpf="123.456.789-00"
)

cliente_2 = Cliente(
    id=2,
    nome="João",
    cpf="987.654.321-00"
)


# Criando duas contas

conta_1 = Conta(
    numero=1001,
    cliente=cliente_1
)

conta_2 = Conta(
    numero=1002,
    cliente=cliente_2
)


# Realizando operações

conta_1.depositar("1500,00")
conta_1.sacar("200,00")
conta_1.transferir(conta_2, "350,50")


# Exibindo os extratos

conta_1.exibir_extrato()
conta_2.exibir_extrato()

Depósito de R$ 1.500,00 realizado com sucesso.
Saque de R$ 200,00 realizado com sucesso.
Transferência de R$ 350,50 realizada com sucesso.
EXTRATO DA CONTA 1001
Titular: Guilherme
CPF: 123.456.789-00
Status: ATIVA
16/09/2026 às 17:21:51 | DEPÓSITO                  |     R$ 1.500,00 | Saldo:     R$ 1.500,00 | Depósito realizado
16/09/2026 às 17:21:51 | SAQUE                     |      R$ -200,00 | Saldo:     R$ 1.300,00 | Saque realizado
16/09/2026 às 17:21:51 | TRANSFERÊNCIA ENVIADA     |      R$ -350,50 | Saldo:       R$ 949,50 | Transferência enviada para a conta 1002, titular João
Saldo atual: R$ 949,50
EXTRATO DA CONTA 1002
Titular: João
CPF: 987.654.321-00
Status: ATIVA
16/09/2026 às 17:21:51 | TRANSFERÊNCIA RECEBIDA    |       R$ 350,50 | Saldo:       R$ 350,50 | Transferência recebida da conta 1001, titular Guilherme
Saldo atual: R$ 350,50


banco.db

In [ ]:
#conexao = sqlite3.connect("banco.db")

#cursor = conexao.cursor()
#cursor.execute ("""
#CREATE TABLE IF NOT EXISTS clientes (
 #   id INTEGER PRIMARY KEY,
  #  nome TEXT NOT NULL,
   # cpf TEXT NOT NULL UNIQUE
#)
#""")

In [ ]:
#Tabela de clientes
#cursor.execute ("""
#CREATE TABLE IF NOT EXISTS clientes (
#    id INTEGER PRIMARY KEY,
#    nome TEXT NOT NULL,
#     cpf TEXT NOT NULL UNIQUE
#)
#""")

In [ ]:
#tabela de contas no banco

cursor.execute("""
CREATE TABLE IF NOT EXISTS contas (
    numero INTEGER PRIMARY KEY,
    cliente_id INTEGER,
    saldo REAL DEFAULT 0,

    FOREIGN KEY(cliente_id)
    REFERENCES clientes(id)
)
""")

In [ ]:
#tabela de movimentações
cursor.execute("""
CREATE TABLE IF NOT EXISTS movimentacoes (

    id INTEGER PRIMARY KEY AUTOINCREMENT,

    conta_numero INTEGER,

    tipo TEXT,

    valor REAL,

    data TEXT,

    descricao TEXT,

    FOREIGN KEY(conta_numero)
    REFERENCES contas(numero)
)
""")


In [ ]:
#Confirmação de criação
conexao.commit()
print("Banco criado com sucesso")

In [ ]:
# teste de criação de cliente novo
banco = Banco()

cliente = banco.cadastrar_cliente(
    "Fabiano",
    "14445678922"1
)

conta = banco.criar_conta(cliente.id)

print(conta)

In [ ]:
#cursor.execute("""
#DELETE FROM clientes
#""")

#conexao.commit()

In [ ]:
#teste de consulta de clientes
cursor.execute(
    "SELECT * FROM clientes"
)

clientes = cursor.fetchall()

for cliente in clientes:
    print(cliente)

In [ ]:
#Criação de conta
cursor.execute(
    """
    INSERT INTO contas
    (numero, cliente_id)

    VALUES (?, ?)
    """,
    (1003, 3)
)

conexao.commit()
#criei duas contas

In [9]:
cursor.execute("DELETE FROM clientes")
cursor.execute("DELETE FROM contas")

conexao.commit()

NameError: name 'cursor' is not defined

In [ ]:
#teste consulta de conta
#cursor.execute(
 #   "SELECT * FROM contas"
#)

#print(cursor.fetchall())

In [ ]:
# quais clientes já estão no banco
#cursor.execute("""
#SELECT * FROM clientes
#""")

#for cliente in cursor.fetchall():
 #   print(cliente)